# 01 — Corpus scan

Reproduces **Method 2.2** of the report: extracts from `EleutherAI/the_pile_deduplicated`

1. **All verified arithmetic expressions** `a op b = c` (op ∈ {+, −, ×, ÷}), yielding
   `data/pile_arith_matches_full_fresh_deduped.jsonl` (≈ 204,073 expressions as reported).
2. **Every standalone integer** for the number-distribution analyses, yielding
   `data/pile_number_distribution.json`.

These scripts are also packaged under `scripts/` for SLURM submission:

- `scripts/scan_pile_local.py` — pile arith expressions (streaming, resumable).
- `scripts/scan_numbers.py` — integer counts (streaming, resumable).
- `scripts/scan_numbers.sbatch` and `scripts/scan_pile_local.sbatch` — SLURM job files.

**Note:** both scripts stream the dataset, so a full run takes hours/days on a single GPU node.
The pre-computed JSONL/JSON outputs are checked in under `data/` so the rest of the
project runs without re-scanning. Re-run this notebook only to extend the scan to more docs
or to verify reproducibility.

In [2]:
# Cell 1 — point HF caches at the local store (matches 00_setup.ipynb).
import os

ROOT = os.getcwd()
while ROOT != "/" and not os.path.isdir(os.path.join(ROOT, "hf", "hub")):
    ROOT = os.path.dirname(ROOT)
assert os.path.isdir(os.path.join(ROOT, "hf", "hub")), "could not locate <root>/hf/hub"

os.environ["HF_HOME"] = os.path.join(ROOT, "hf")
os.environ["HF_HUB_CACHE"] = os.path.join(ROOT, "hf", "hub")
os.environ["HF_DATASETS_CACHE"] = os.path.join(ROOT, "hf", "datasets")
os.environ['PATH'] += ':/storage/home/hcoda1/9/kzhang430/.local/bin'
print("ROOT =", ROOT)
print("HF_HOME =", os.environ["HF_HOME"])

ROOT = /storage/scratch1/9/kzhang430
HF_HOME = /storage/scratch1/9/kzhang430/hf


## A. Arithmetic expressions in the Pile

Run the local scan script (writes to `../data/pile_arith_matches_full_fresh_deduped.jsonl`).

```bash
cd <scratch_root>/notebooks       # same CWD the notebooks assume
python ../scripts/scan_pile_local.py \
    --start 0 \
    --out ../data/pile_arith_matches_full_fresh_deduped.jsonl \
    --flush_every 2000
```

Or via SLURM, from the scratch root:

```bash
sbatch scripts/scan_pile_local.sbatch
```

## B. Standalone integers in the Pile

Same idea, integer-only scan:

```bash
cd <scratch_root>
python scripts/scan_numbers.py \
    --dataset EleutherAI/the_pile_deduplicated \
    --split train \
    --start 0 \
    --out data/pile_number_distribution.json \
    --flush_every 20000
```

The output JSON contains three aggregates:

- `digit_counts` — number of distinct integers of each digit length.
- `log10_hist` — binned `log10(|x|+1)` histogram (1000 bins, 0–50).
- `small_value_counts` — exact integer counts for `|x| ≤ 10000`.

## C. Verify the cached outputs

Once you've either scanned or are using the cached files, run the cells below to confirm
the JSONL/JSON files exist and report the headline numbers from the report.

In [3]:
import os, json

arith_path = "../data/pile_arith_matches_full_fresh_deduped.jsonl"
num_path = "../data/pile_number_distribution.json"

print("==== Verified arith expressions ====")
if os.path.exists(arith_path):
    n_lines = sum(1 for _ in open(arith_path))
    size_mb = os.path.getsize(arith_path) / 1e6
    print(f"  {arith_path}: {n_lines:,} lines, {size_mb:.1f} MB")
    print(f"  (report quotes 204,073 verified expressions)")
else:
    print(f"  MISSING: {arith_path} — run the scan above to produce it.")

print()
print("==== Integer aggregates ====")
if os.path.exists(num_path):
    with open(num_path) as f:
        d = json.load(f)
    print(f"  docs_seen        : {d.get('docs_seen', '?'):,}")
    print(f"  total_numbers    : {d.get('total_numbers', '?'):,}")
    print(f"  digit_counts keys: {list(d.get('digit_counts', {}).keys())[:10]} ...")
    print(f"  small_value keys : {list(d.get('small_value_counts', {}).keys())[:10]} ...")
    print(f"  (report quotes ≈ 4.78 billion integers, with counts for 1–100 kept)")
else:
    print(f"  MISSING: {num_path} — run the scan above to produce it.")

==== Verified arith expressions ====
  ../data/pile_arith_matches_full_fresh_deduped.jsonl: 204,073 lines, 17.4 MB
  (report quotes 204,073 verified expressions)

==== Integer aggregates ====
  docs_seen        : 134,318,121
  total_numbers    : 4,776,629,260
  digit_counts keys: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10'] ...
  small_value keys : ['-10000', '-9999', '-9998', '-9997', '-9996', '-9995', '-9994', '-9993', '-9992', '-9991'] ...
  (report quotes ≈ 4.78 billion integers, with counts for 1–100 kept)


## D. Where the next notebooks consume these

- `02_number_distribution.ipynb` (Fig 1, Fig 4) — reads `data/pile_number_distribution.json`.
- `03_metrics_powerlaw.ipynb` (Fig 2, Fig 3) — reads
  `data/pile_arith_matches_full_fresh_deduped.jsonl`.
- `04_arithmetic_main.ipynb` — reads both, plus uses the Pythia sweeps in
  `data/pythia_freq_results*.csv` and the fewshot / multi generations.

No model is loaded in this notebook — the scan uses only the `datasets` streaming API.